In [10]:
import cogent3
from cogent3 import get_app
from cogent3 import load_aligned_seqs
from cogent3.maths.matrix_exponential_integration import expected_number_subs

import paths
import libs


# Calculating Intergenic Ancestral Repeat Q

In [15]:
relative_folder_in = "intergenicAR/alldata_chrm22/" 
folder_in = paths.DATA_HUMCHIMPGOR116 + relative_folder_in

sequence = 'homo_sapiens-22-36038797-36056389.fa'

omit_degs_noncds = get_app("omit_degenerates", moltype="dna", motif_length=3)
rename_noncds = libs.renamer_noncds_aligned()

aln_igar = load_aligned_seqs(filename = folder_in + sequence, moltype='dna')
aln_igar = rename_noncds(aln_igar)
aln_igar = omit_degs_noncds(aln_igar)
aln_igar


NotCompleted(type=FAIL, origin=omit_degenerates, source="homo_sapiens-22-36038797-36056389", message="all columns contained degenerates")

In [ ]:
aln_igar = omit_degs_noncds(aln_igar, motif_length=3)
GN_subsmodel = get_app("model", "GN", time_het="max", lf_args={"discrete_edges": ["Gorilla"]}, optimise_motif_probs=True, show_progress=True)
result_IGAR = GN_subsmodel(aln_igar)

humanQ_IGAR = result_IGAR.lf.get_rate_matrix_for_edge("Human", calibrated=False)

AttributeError: 'NotCompleted' object has no attribute 'lf'

In [12]:
result_IGAR

NotCompleted(type=FAIL, origin=omit_degenerates, source="homo_sapiens-22-36038797-36056389", message="all columns contained degenerates")

In [8]:
result_IGAR.lf

AttributeError: 'NotCompleted' object has no attribute 'lf'

# Calculating cds ENS

In [ ]:
#folder_in = paths.DATA_APES114 + 'cds/codon_aligned/'
relative_folder_in = "cds/alldata_chrm22/" 
folder_in = paths.DATA_HUMCHIMPGOR116 + relative_folder_in

sequence = 'ENSG00000133466.fa'

omit_degs_cds = get_app("omit_degenerates", moltype="dna", motif_length=3)

aln_cds = load_aligned_seqs(filename = folder_in + sequence, moltype='dna')
omit_degs_cds(aln_cds)

result_cds = GN_subsmodel(aln_cds)

cds_motif_probs = result_cds.lf.get_param_value("mprobs")
humanQ_cds = result_cds.lf.get_rate_matrix_for_edge("Human", calibrated=False)
humanENS_cds = expected_number_subs(cds_motif_probs, humanQ_cds, t=1.0)
#This is the ENS of cds if selection was the same as on IGAR. Instead of using the cds Q matrix humanQ_cds, I use the IGAR Q humanQ_IGAR
humanENS_cds_IGARQ = expected_number_subs(cds_motif_probs, humanQ_IGAR, t=1.0)


print("Human ENS cds: ")
print(humanENS_cds)
print("Human ENS cds with IGAR Q: ")
print(humanENS_cds_IGARQ)

#result_cds.lf

   0%|          |00:00<?

   0%|          |00:00<?

Human ENS cds: 
0.004810325939899154
Human ENS cds with IGAR Q: 
0.008025096693524934


# Calculating Intergenic ENS

In [7]:
#folder_in = paths.DATA_APES114 + 'cds/codon_aligned/'
folder_in = 'ig/'

sequence = 'homo_sapiens-22-35347992-35372172.fa'

aln_ig = load_aligned_seqs(filename = folder_in + sequence, moltype='dna')
aln_ig = omit_degs_noncds(aln_ig)

result_ig = GN_subsmodel(aln_ig)

ig_motif_probs = result_ig.lf.get_param_value("mprobs")
humanQ_ig = result_ig.lf.get_rate_matrix_for_edge("Human", calibrated=False)
humanENS_ig = expected_number_subs(ig_motif_probs, humanQ_ig, t=1.0)
#This is the ENS of cds if selection was the same as on IGAR. Instead of using the cds Q matrix humanQ_cds, I use the IGAR Q humanQ_IGAR
humanENS_ig_IGARQ = expected_number_subs(ig_motif_probs, humanQ_IGAR, t=1.0)


print("Human ENS IG: ")
print(humanENS_ig)
print("Human ENS IG with IGAR Q: ")
print(humanENS_ig_IGARQ)

   0%|          |00:00<?

   0%|          |00:00<?

Human ENS IG: 
0.006526356909947123
Human ENS IG with IGAR Q: 
0.007860543750877583


# Calculating cds constraint

In [4]:
cds_constraint = (humanENS_cds_IGARQ - humanENS_cds)/humanENS_cds_IGARQ
cds_constraint

np.float64(0.4005896597133371)

In [8]:
ig_constraint = (humanENS_ig_IGARQ - humanENS_ig)/humanENS_ig_IGARQ
ig_constraint

np.float64(0.16973213090780215)